In [ ]:
from glob import glob
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from factor_analyzer import FactorAnalyzer
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.preprocessing import StandardScaler

In [ ]:
files = glob("/data/uscuni-restricted/04_spatial_census/*nadzsj*")
files.sort()

In [ ]:
for file in files:
    df = gpd.read_parquet(file)
    if "Obyvatelstvo celkem" in df.columns:
        df = df.drop(columns="Obyvatelstvo celkem")
        df.to_parquet(file)

In [ ]:
total_pop = gpd.read_parquet(files[9])

In [ ]:
# total = total_pop[["nadzsjd", "Obyvatelstvo celkem"]].set_index("nadzsjd")

In [ ]:
# total.to_csv("/data/uscuni-restricted/04_spatial_census/total.csv", index=True)

In [ ]:
total_pop["Obyvatelstvo - věk: 0 - 6  - celkem"] = total_pop[
    [
        "Obyvatelstvo - věk: 0 - 2  - celkem",
        "Obyvatelstvo - věk: 3 - 4  - celkem",
        "Obyvatelstvo - věk: 5  - 6 - celkem",
    ]
].sum(axis=1)
total_pop["Obyvatelstvo - věk: 7 - 14  - celkem"] = total_pop[
    ["Obyvatelstvo - věk: 7 - 9  - celkem", "Obyvatelstvo - věk: 10 - 14  - celkem"]
].sum(axis=1)
total_pop["Obyvatelstvo - věk: 15 - 24  - celkem"] = total_pop[
    [
        "Obyvatelstvo - věk: 15 - 17  - celkem",
        "Obyvatelstvo - věk: 18 - 19  - celkem",
        "Obyvatelstvo - věk: 20 - 24  - celkem",
    ]
].sum(axis=1)
total_pop["Obyvatelstvo - věk: 25 - 34  - celkem"] = total_pop[
    ["Obyvatelstvo - věk: 25 - 29  - celkem", "Obyvatelstvo - věk: 30 - 34  - celkem"]
].sum(axis=1)
total_pop["Obyvatelstvo - věk: 35 - 44  - celkem"] = total_pop[
    [
        "Obyvatelstvo - věk: 35 - 39  - celkem",
        "Obyvatelstvo - věk: 40 - 44  - celkem",
    ]
].sum(axis=1)
total_pop["Obyvatelstvo - věk: 45 - 54  - celkem"] = total_pop[
    [
        "Obyvatelstvo - věk: 45 - 49  - celkem",
        "Obyvatelstvo - věk: 50 - 54  - celkem",
    ]
].sum(axis=1)
total_pop["Obyvatelstvo - věk: 55 - 64  - celkem"] = total_pop[
    ["Obyvatelstvo - věk: 55 - 59  - celkem", "Obyvatelstvo - věk: 60 - 64  - celkem"]
].sum(axis=1)
total_pop["Obyvatelstvo - věk: 65 - 74  - celkem"] = total_pop[
    [
        "Obyvatelstvo - věk: 65 - 69  - celkem",
        "Obyvatelstvo - věk: 70 - 74  - celkem",
    ]
].sum(axis=1)
total_pop["Obyvatelstvo - věk: 75 - 84  - celkem"] = total_pop[
    [
        "Obyvatelstvo - věk: 75 - 79  - celkem",
        "Obyvatelstvo - věk: 80 - 84  - celkem",
    ]
].sum(axis=1)

In [ ]:
total_pop = total_pop.drop(
    columns=total_pop.loc[
        :, "Obyvatelstvo - věk: 0 - 2  - celkem":"Obyvatelstvo - věk: 80 - 84  - celkem"
    ].columns
)

In [ ]:
# total_pop = total_pop.drop(columns="Obyvatelstvo celkem")

In [ ]:
total_pop.to_parquet(
    "/data/uscuni-restricted/04_spatial_census/nadzsjd_pop_age_gender_2021.parquet"
)

In [ ]:
total = pd.read_csv(
    "/data/uscuni-restricted/04_spatial_census/total.csv",
    dtype={"nadzsjd": str},
    index_col=0,
)

In [ ]:
area = gpd.read_parquet(files[4])

In [ ]:
density = (
    total["Obyvatelstvo celkem"]
    / area["Celková plocha obydlených bytů v m2 v domech celkem"]
)
density = density.fillna(0)

In [ ]:
area["Hustota obyvatel na obytnou plochu"] = density
area = area.drop(columns="Celková plocha obydlených bytů v m2 v domech celkem")

In [ ]:
area.to_parquet(
    "/data/uscuni-restricted/04_spatial_census/nadzsjd_housing_size_facilities_2021.parquet"
)

In [ ]:
flats = gpd.read_parquet(files[6])

In [ ]:
poeple_flats = total["Obyvatelstvo celkem"] / flats["Byty v domech celkem"]
poeple_flats = poeple_flats.fillna(0)

In [ ]:
flats["Počet obyvatel na byt"] = poeple_flats
flats = flats.drop(columns="Byty v domech celkem")
flats.to_parquet(
    "/data/uscuni-restricted/04_spatial_census/nadzsjd_housing_flats_2021.parquet"
)

In [ ]:
houses = gpd.read_parquet(files[7])

In [ ]:
poeple_houses = total["Obyvatelstvo celkem"] / houses["Domy celkem"]
poeple_houses = poeple_houses.fillna(0)

In [ ]:
houses["Počet obyvatel na dům"] = poeple_houses
houses = houses.drop(columns="Domy celkem")
houses.to_parquet(
    "/data/uscuni-restricted/04_spatial_census/nadzsjd_housing_houses_2021.parquet"
)

In [ ]:
houses